# Section 1 — contract destination starter

**Varshini: change destination only.** Roll timing, capacity handling, rounding, target weights, the common benchmark, accounting and metric stay fixed. Choose a destination only at a scheduled baseline roll event; do not reselect daily.

[Part 2 Issue #12](https://github.com/JacopoMichelacci/Futures_Class/issues/12), Version 8 §§39–42: next deferred (baseline), second deferred, closest to 45 sessions, closest to 75 sessions. These alternatives are TODOs, not implemented below.

Run all cells from a fresh Python kernel using the project's existing uv environment. These notebooks use only the shared synthetic fixture and standard-library Python, with no online access. Outputs are deliberately cleared before sharing.

**Part 2 responsibility:** each teammate implements their own reusable engine and substantive tests, opens their own issue-linked PR, and reviews teammate PRs. This starter supplies inputs and interface examples, not the strand solution. Keep engine logic in `src/` and tests in `tests/unit/`. No empirical conclusions or preferred-method selection belong here. Do not consume another strand's preferred results.

See [Section 0 interfaces](../docs/section0_phase_a.md) and [fixture provenance](../data/fixtures/README.md). Phase B still needs sourced first-notice metadata, verified exchange/shortened-session times, historical benchmark/baseline quantities, production tables, the common 100-date/70% gate and AUM/capacity preflight. The synthetic calendar is not a real CBOT calendar. Daily return Tracking RMSE in basis points remains the sole ranking metric for the later historical comparison.

In [ ]:
from pathlib import Path
from decimal import Decimal
from pprint import pprint
import sys

# Start the kernel in this repository or the notebook's own folder.
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "config/project.yaml").is_file()
             and (p / "data/fixtures/phase_a.json").is_file()
             and (p / "src/__init__.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start the kernel in the repository or this notebook's folder.")
sys.path.insert(0, str(ROOT))
import src
assert Path(src.__file__).resolve().parent == ROOT / "src", "Restart the kernel for this checkout."
from src.validation.schemas import load_project_config, HOLDINGS_COLUMNS
from src.data.fixtures import load_fixture

config = load_project_config(ROOT / "config/project.yaml")
f = load_fixture(ROOT / "data/fixtures/phase_a.json",
                 config["max_liquidity_source_age_sessions"])
return_date = f.raw["return_dates"][0]
context = f.calendar.decision(return_date)
print("SYNTHETIC FIXTURES ONLY", config["roots"], "return interval ends", return_date)


### Inspect the shared baseline event and its decision-time curve

Use the shared generator to obtain the outgoing contract and roll date. The relevant curve below is from that event's decision session. Eligibility uses the supplied delivery-avoidance anchor; expiration is not substituted.

In [ ]:
from src.portfolio.baseline import build_baseline_path
case = f.raw["baseline_path_case"]
path = build_baseline_path(case["eligible_curve"], case["decision_dates"],
                           case["commodities"], f.calendar, f.catalogue)
event = path.roll_events[0]
pprint(event)
pprint(path.positions[:3])


In [ ]:
outgoing = f.catalogue[event["outgoing_contract"]]
event_curve = [r for r in case["eligible_curve"]
               if r["date"] == event["date"] and r["commodity"] == event["commodity"]]
# Inspection only: no alternative destination is selected here.
deferred_rows = [r for r in event_curve if r["eligible"] and r["anchor"] > outgoing.anchor]
pprint(sorted(deferred_rows, key=lambda r: (r["anchor"], r["contract"])))


### Expected rule output and work to complete

An illustrative destination record contains the unchanged decision/event date, commodity, outgoing contract, selected exact contract and an explicit fallback flag. This is a suggested adapter shape, not a new production schema. The baseline example below comes directly from Section 0; an unavailable target-day destination must remain unimplemented and flagged, not silently replaced.

**TODO Varshini:** implement and test `deferred_candidates`, `select_next_deferred`, `select_second_deferred`, `select_target_days_deferred`, `build_destination_schedule` and `apply_destination_rule_to_baseline`. Cover later/eligible candidates, 45/75-session ties (nearer maturity), the second-deferred single-candidate fallback, no-candidate handling and no switches between roll events. Keep the -5-session timing fixed. Open your own issue-linked PR.

In [ ]:
baseline_destination_example = {
    "date": event["date"], "commodity": event["commodity"],
    "outgoing_contract": event["outgoing_contract"],
    "selected_contract": event["baseline_destination"], "fallback_flag": False,
}
pprint(baseline_destination_example)
# TODO: load your tested destination module once it exists in this checkout.
alternative_destination_output = None


### Integration boundary: future engine output → shared accounting

The TODO below is intentionally inactive. Your engine must produce canonical `Holding` rows for the baseline and all three alternatives on the same fixture dates, plus truthful decision-validity flags. Each row's `date` is the end of `(t-1,t]`; quantities and inputs are fixed by the preceding cut-off. Preserve exact contracts and explicit zero positions. Decision timestamps cannot be invented from later valuation marks.

Use the supplied fixture benchmark once and call the shared engine once for all compared methods. Missing required settlements invalidate the common date; never fill them with zero or splice maturities. One fixed AUM divides every portfolio's P&L. This adapter is for the synthetic accounting example only; the hand-authored benchmark quantities are not a constructed 60-session portfolio. Phase B must supply the actual common benchmark and frozen comparison sample.

In [ ]:
from src.portfolio.holdings import HoldingsBook
from src.metrics.portfolio_pnl import portfolio_pnl
from src.metrics.tracking_rmse_bps import tracking_rmse_bps

future_holding_rows = None  # TODO: baseline + three alternatives, produced by your engine.
future_decision_validity = None  # TODO: {(date, strategy): bool}, including the benchmark.
print("Canonical fields:", HOLDINGS_COLUMNS)
if future_holding_rows is None or future_decision_validity is None:
    print("TODO adapter: no strategy implemented or evaluated by this starter.")
else:
    benchmark_rows = [r for r in f.holdings.rows if r.strategy == f.raw["benchmark_strategy"]]
    strategies = [f.raw["benchmark_strategy"]] + sorted({r.strategy for r in future_holding_rows})
    assert len(strategies) == len(set(strategies)) == 5, "Provide baseline and three distinct alternatives."
    book = HoldingsBook(benchmark_rows + list(future_holding_rows), f.calendar, f.catalogue)
    daily = portfolio_pnl(book, f.observations, f.raw["return_dates"], strategies,
                          f.raw["fixed_aum"], f.raw["snapshot_end_exclusive"],
                          decision_validity=future_decision_validity)
    pprint(tracking_rmse_bps(daily))  # Synthetic arithmetic check; no preferred-method selection.
